# Imports

In [11]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [12]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [13]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet')

In [14]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.004831,0.002142,0.993027,0.005220,0.002128,0.992652,0.005655,0.000463,0.993882
1,0.998229,0.000143,0.001628,0.997774,0.000322,0.001904,0.994638,0.000368,0.004994
2,0.003982,0.000584,0.995434,0.002362,0.000710,0.996928,0.002327,0.000919,0.996754
3,0.007424,0.001954,0.990622,0.002922,0.003628,0.993450,0.003433,0.002374,0.994194
4,0.993583,0.000012,0.006405,0.997151,0.000013,0.002836,0.994112,0.000005,0.005883


In [15]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.011986,0.002516,0.985498,0.009749,0.003956,0.986295,0.008185,0.005526,0.986288
1,0.469590,0.002922,0.527488,0.344531,0.002104,0.653365,0.530403,0.002094,0.467503
2,0.998492,0.000991,0.000517,0.998555,0.001132,0.000313,0.996306,0.003216,0.000478
3,0.986619,0.000009,0.013372,0.978274,0.000024,0.021702,0.990683,0.000093,0.009224
4,0.008520,0.001975,0.989505,0.005601,0.002080,0.992319,0.003507,0.001947,0.994546


# Machine Learning

In [16]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [17]:
cols_use = ['lgbm_0', 'lgbm_1', 'lgbm_2']

In [28]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.0, 10.0, log=False)
    w1 = trial.suggest_float('weight_class_1', 0.0, 10.0, log=False)
    w2 = trial.suggest_float('weight_class_2', 0.0, 10.0, log=False)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, cols_use].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=1000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-03 09:48:21,824] A new study created in memory with name: no-name-b0a35d82-bc19-403a-8c20-11a160a6ac89
                                                                                                                                                                                                                   

[I 2026-07-03 09:48:21,952] Trial 1 finished with value: 0.8750860235668175 and parameters: {'weight_class_0': 7.634070141611728, 'weight_class_1': 5.288818933638407, 'weight_class_2': 8.789810164072525}. Best is trial 1 with value: 0.8750860235668175.
[I 2026-07-03 09:48:21,974] Trial 0 finished with value: 0.922868217921903 and parameters: {'weight_class_0': 2.241667946644789, 'weight_class_1': 7.237902222124873, 'weight_class_2': 3.759546990559394}. Best is trial 0 with value: 0.922868217921903.
[I 2026-07-03 09:48:21,976] Trial 4 finished with value: 0.8638407212226081 and parameters: {'weight_class_0': 5.239714043257173, 'weight_class_1': 2.427114508885204, 'weight_class_2': 5.546624144247662}. Best is trial 0 with value: 0.922868217921903.
[I 2026-07-03 09:48:21,981] Trial 3 finished with value: 0.9093772956041644 and parameters: {'weight_class_0': 4.958326271934564, 'weight_class_1': 9.190359929442792, 'weight_class_2': 7.495432097512596}. Best is trial 0 with value: 0.922868217

[I 2026-07-03 09:48:22,078] Trial 12 finished with value: 0.8685604399699646 and parameters: {'weight_class_0': 5.92271756682212, 'weight_class_1': 5.51366423364957, 'weight_class_2': 4.068405364046537}. Best is trial 7 with value: 0.9400732809529128.
[I 2026-07-03 09:48:22,116] Trial 15 finished with value: 0.9474616188720312 and parameters: {'weight_class_0': 0.7904031631165931, 'weight_class_1': 6.879179920024453, 'weight_class_2': 4.03549443884657}. Best is trial 15 with value: 0.9474616188720312.
[I 2026-07-03 09:48:22,123] Trial 13 finished with value: 0.8392107039418418 and parameters: {'weight_class_0': 8.780745788073045, 'weight_class_1': 2.197775703385326, 'weight_class_2': 5.101777774067054}. Best is trial 15 with value: 0.9474616188720312.
[I 2026-07-03 09:48:22,128] Trial 14 finished with value: 0.855246858971391 and parameters: {'weight_class_0': 7.801202701148328, 'weight_class_1': 2.110222929856138, 'weight_class_2': 8.532722518851156}. Best is trial 15 with value: 0.94

Best trial: 39. Best value: 0.948914:   4%|█████                                                                                                                                 | 38/1000 [00:00<00:16, 58.10it/s]

[I 2026-07-03 09:48:22,331] Trial 26 finished with value: 0.8862085920547229 and parameters: {'weight_class_0': 3.111750737907646, 'weight_class_1': 6.230103346023241, 'weight_class_2': 1.8409800799498912}. Best is trial 15 with value: 0.9474616188720312.
[I 2026-07-03 09:48:22,338] Trial 27 finished with value: 0.8857732556926035 and parameters: {'weight_class_0': 3.162444200876619, 'weight_class_1': 6.411849265692412, 'weight_class_2': 1.7524661143292368}. Best is trial 15 with value: 0.9474616188720312.
[I 2026-07-03 09:48:22,348] Trial 28 finished with value: 0.8952861599934111 and parameters: {'weight_class_0': 3.0401310577418865, 'weight_class_1': 8.384377901963393, 'weight_class_2': 2.141003012916972}. Best is trial 15 with value: 0.9474616188720312.
[I 2026-07-03 09:48:22,359] Trial 29 finished with value: 0.8932947061211998 and parameters: {'weight_class_0': 2.915486132275584, 'weight_class_1': 6.195743229169758, 'weight_class_2': 2.3516235159286865}. Best is trial 15 with val

...

[I 2026-07-03 09:48:39,345] Trial 965 finished with value: 0.9482146502382194 and parameters: {'weight_class_0': 0.8510037550721825, 'weight_class_1': 5.717466118767591, 'weight_class_2': 5.764568594393284}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,409] Trial 968 finished with value: 0.9484863816189716 and parameters: {'weight_class_0': 0.8800585632520422, 'weight_class_1': 7.739535343445325, 'weight_class_2': 5.630436149494881}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,412] Trial 967 finished with value: 0.9487401576326739 and parameters: {'weight_class_0': 0.8522682379010267, 'weight_class_1': 7.763277673029913, 'weight_class_2': 5.7234404394543885}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,446] Trial 969 finished with value: 0.8298042109778243 and parameters: {'weight_class_0': 0.025412479172698788, 'weight_class_1': 7.8463878188795135, 'weight_class_2': 5.754893759911023}. Best is trial 560

Best trial: 560. Best value: 0.949962:  99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏ | 986/1000 [00:18<00:00, 48.34it/s]

[I 2026-07-03 09:48:39,603] Trial 977 finished with value: 0.9493120762022231 and parameters: {'weight_class_0': 0.693780437390592, 'weight_class_1': 7.488475065244823, 'weight_class_2': 5.555866246514391}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,660] Trial 979 finished with value: 0.9461505859285374 and parameters: {'weight_class_0': 0.696226903124699, 'weight_class_1': 7.532926542331809, 'weight_class_2': 2.742602108090694}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,661] Trial 978 finished with value: 0.9493647638033226 and parameters: {'weight_class_0': 0.6768622754698883, 'weight_class_1': 7.514095937537275, 'weight_class_2': 5.479690825089807}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,673] Trial 980 finished with value: 0.9495419063787062 and parameters: {'weight_class_0': 0.6511127226196651, 'weight_class_1': 7.47524401578641, 'weight_class_2': 5.57580964567061}. Best is trial 560 with va

Best trial: 560. Best value: 0.949962: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:18<00:00, 55.01it/s]

[I 2026-07-03 09:48:39,814] Trial 986 finished with value: 0.9491818249702922 and parameters: {'weight_class_0': 0.28484277522484225, 'weight_class_1': 6.704576159151015, 'weight_class_2': 5.432071461035777}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,831] Trial 988 finished with value: 0.949082026134677 and parameters: {'weight_class_0': 0.2693434867599558, 'weight_class_1': 6.727763899693121, 'weight_class_2': 5.391402845248545}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,841] Trial 989 finished with value: 0.9493279501131547 and parameters: {'weight_class_0': 0.2981370920048505, 'weight_class_1': 6.633251930165377, 'weight_class_2': 5.357865298686301}. Best is trial 560 with value: 0.949961556182569.
[I 2026-07-03 09:48:39,912] Trial 991 finished with value: 0.948916273876177 and parameters: {'weight_class_0': 0.29450831216608303, 'weight_class_1': 6.682966219133949, 'weight_class_2': 6.639790593542927}. Best is trial 560 wit

In [29]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, cols_use].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [30]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [31]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.1.0_lightgbm_submission.csv', index=False)

In [32]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
